In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# Load Dataset

In [2]:
train_df = pd.read_csv("/content/fraudTrain.csv")
test_df = pd.read_csv("/content/fraudTest.csv")

print(train_df.shape)
print(test_df.shape)

(62256, 23)
(70004, 23)


In [42]:
train_df.head()

,cc_num,merchant,category,amt,gender,city,state,zip,lat,long,...,job,unix_time,merch_lat,merch_long,is_fraud,year,month,day,hour,age
0,2703186189652095,1154,12,4.97,0,441,19,28654,36.0788,-81.1781,...,288,1.325376e+09,36.011293,-82.048315,0,2019,1,1,0,31
1,630423337322,780,8,107.23,0,525,41,99160,48.8878,-118.2105,...,350,1.325376e+09,49.159047,-118.186462,0,2019,1,1,0,41
2,38859492057661,984,0,220.11,1,382,4,83252,42.1808,-112.2620,...,219,1.325376e+09,43.150704,-112.154481,0,2019,1,1,0,57
3,3534093764340240,944,6,45.00,1,789,18,59632,46.2306,-112.1138,...,243,1.325376e+09,47.034331,-112.561071,0,2019,1,1,0,52
4,375534208663984,857,13,41.96,1,118,39,24433,38.4207,-79.4629,...,15,1.325376e+09,38.674999,-78.632459,0,2019,1,1,0,33


In [3]:
for df in [train_df, test_df]:

    df['trans_date_trans_time'] = pd.to_datetime(
        df['trans_date_trans_time']
    )

    df['year'] = df['trans_date_trans_time'].dt.year
    df['month'] = df['trans_date_trans_time'].dt.month
    df['day'] = df['trans_date_trans_time'].dt.day
    df['hour'] = df['trans_date_trans_time'].dt.hour

    df['dob'] = pd.to_datetime(df['dob'])

    df['age'] = (
        df['trans_date_trans_time'].dt.year
        - df['dob'].dt.year
    )

#Remove Unnecessary Columns

In [4]:
drop_cols = [
    'Unnamed: 0',
    'trans_date_trans_time',
    'first',
    'last',
    'street',
    'trans_num',
    'dob'
]

train_df.drop(columns=drop_cols, inplace=True)
test_df.drop(columns=drop_cols, inplace=True)

In [24]:
# Explicitly list all categorical columns that need encoding
cat_cols = ['merchant', 'category', 'gender', 'job', 'city', 'state']

# 'is_fraud' is the target variable and should not be encoded as a feature.

encoders = {}

for col in cat_cols:

    le = LabelEncoder()

    # Combine unique values from both train and test data (after filling NaNs) for fitting
    combined_data = pd.concat([
        train_df[col].fillna('Missing').astype(str),
        test_df[col].fillna('Missing').astype(str)
    ]).unique()

    le.fit(combined_data)

    train_df[col] = le.transform(
        train_df[col].fillna('Missing').astype(str)
    )

    test_df[col] = le.transform(
        test_df[col].fillna('Missing').astype(str)
    )

    encoders[col] = le

In [25]:
# Ensure 'is_fraud' is numeric, coercing errors to NaN
train_df['is_fraud'] = pd.to_numeric(train_df['is_fraud'], errors='coerce')
test_df['is_fraud'] = pd.to_numeric(test_df['is_fraud'], errors='coerce')

# Drop rows where 'is_fraud' is NaN from both train_df and test_df
train_df.dropna(subset=['is_fraud'], inplace=True)
test_df.dropna(subset=['is_fraud'], inplace=True)

# Convert 'is_fraud' to integer type after ensuring no NaNs
train_df['is_fraud'] = train_df['is_fraud'].astype(int)
test_df['is_fraud'] = test_df['is_fraud'].astype(int)

X_train = train_df.drop('is_fraud', axis=1)
y_train = train_df['is_fraud']

X_test = test_df.drop('is_fraud', axis=1)
y_test = test_df['is_fraud']

In [26]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=15, n_jobs=-1,
                       random_state=42)

#Predictions

In [27]:
y_pred = rf.predict(X_test)

y_prob = rf.predict_proba(X_test)[:,1]

In [28]:
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print(
    "\nROC-AUC Score:",
    roc_auc_score(y_test, y_prob)
)

Confusion Matrix
[[69725     0]
 [  278     0]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     69725
           1       0.00      0.00      0.00       278

    accuracy                           1.00     70003
   macro avg       0.50      0.50      0.50     70003
weighted avg       0.99      1.00      0.99     70003


ROC-AUC Score: 0.9111142695739428


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [29]:
importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance.head(15))

       Feature  Importance
3          amt    0.466092
18        hour    0.142009
2     category    0.043172
11         job    0.037295
5         city    0.034689
12   unix_time    0.031838
10    city_pop    0.028729
19         age    0.026341
0       cc_num    0.025930
9         long    0.025803
8          lat    0.022850
17         day    0.021345
14  merch_long    0.021156
7          zip    0.019722
13   merch_lat    0.018453


#Logistic Regression Model

In [30]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000
)

lr.fit(X_train, y_train)

pred = lr.predict(X_test)

print(classification_report(
    y_test,
    pred
))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     69725
           1       0.00      0.00      0.00       278

    accuracy                           1.00     70003
   macro avg       0.50      0.50      0.50     70003
weighted avg       0.99      1.00      0.99     70003



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [31]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    max_depth=15,
    class_weight='balanced',
    random_state=42
)

dt.fit(X_train, y_train)

pred = dt.predict(X_test)

print(classification_report(
    y_test,
    pred
))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99     69725
           1       0.00      0.02      0.01       278

    accuracy                           0.98     70003
   macro avg       0.50      0.50      0.50     70003
weighted avg       0.99      0.98      0.99     70003



#Check Fraud Train & Test

In [34]:
sample = X_test.iloc[[0]]

pred = rf.predict(sample)[0]
actual = y_test.iloc[0]

print("Predicted :", "Fraud" if pred == 1 else "Legitimate")
print("Actual    :", "Fraud" if actual == 1 else "Legitimate")

Predicted : Legitimate
Actual    : Legitimate


In [45]:
train_df.head(101)

,cc_num,merchant,category,amt,gender,city,state,zip,lat,long,...,job,unix_time,merch_lat,merch_long,is_fraud,year,month,day,hour,age
0,2703186189652095,1154,12,4.97,0,441,19,28654,36.0788,-81.1781,...,288,1.325376e+09,36.011293,-82.048315,0,2019,1,1,0,31
1,630423337322,780,8,107.23,0,525,41,99160,48.8878,-118.2105,...,350,1.325376e+09,49.159047,-118.186462,0,2019,1,1,0,41
2,38859492057661,984,0,220.11,1,382,4,83252,42.1808,-112.2620,...,219,1.325376e+09,43.150704,-112.154481,0,2019,1,1,0,57
3,3534093764340240,944,6,45.00,1,789,18,59632,46.2306,-112.1138,...,243,1.325376e+09,47.034331,-112.561071,0,2019,1,1,0,52
4,375534208663984,857,13,41.96,1,118,39,24433,38.4207,-79.4629,...,15,1.325376e+09,38.674999,-78.632459,0,2019,1,1,0,33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,4423489490781412,823,6,50.61,1,583,3,52768,41.6858,-90.3706,...,57,1.325380e+09,41.027284,-89.416172,0,2019,1,1,1,62
97,3518234918950662,835,7,50.74,0,669,37,75092,33.6372,-96.6184,...,368,1.325380e+09,34.266941,-96.709668,0,2019,1,1,1,49
98,6593250708747804,1249,8,186.73,0,376,48,33470,26.7383,-80.2760,...,242,1.325380e+09,27.610009,-79.498110,0,2019,1,1,1,42
99,6011492816282597,947,7,39.95,1,676,12,71762,33.3398,-92.7442,...,99,1.325381e+09,33.651123,-91.902918,0,2019,1,1,1,33


In [35]:
row_no = 100

sample = X_test.iloc[[row_no]]

pred = rf.predict(sample)[0]

print("Transaction", row_no)
print("Prediction:", "Fraud" if pred == 1 else "Legitimate")

Transaction 100
Prediction: Legitimate


Demo Test

In [37]:
idx = np.random.randint(0, len(X_test))

sample = X_test.iloc[[idx]]

pred = rf.predict(sample)[0]

print(f"Transaction ID: {idx}")
print("Prediction:", "Fraud" if pred else "Legitimate")

Transaction ID: 29107
Prediction: Legitimate


In [41]:
fraud_idx = y_test[y_test == 1].index[0]

sample = X_test.loc[[fraud_idx]]

pred = rf.predict(sample)[0]
actual = y_test.loc[fraud_idx]

print("Transaction ID:", fraud_idx)
print("Predicted:", "Fraud" if pred == 1 else "Legitimate")
print("Actual   :", "Fraud" if actual == 1 else "Legitimate")

Transaction ID: 1685
Predicted: Legitimate
Actual   : Fraud
